# How to create a simulation

This tutorial demonstrates how to create a simulation.

## What is a simulation?

A simulation contains selected sensors, sources to model ray-trace in space.

## Prerequisites

### Perform imports

In [1]:
from pathlib import Path

from ansys.speos.core import Project, Speos, launcher
from ansys.speos.core.kernel.client import (
    default_docker_channel,
)
from ansys.speos.core.simulation import (
    SimulationInteractive,
    SimulationInverse,
    SimulationVirtualBSDF,
)


### Define constants

The constants help ensure consistency and avoid repetition throughout the example.

In [2]:
HOSTNAME = "localhost"
GRPC_PORT = 50098  # Be sure the Speos GRPC Server has been started on this port.
USE_DOCKER = True  # Set to False if you're running this example locally as a Notebook.
SOURCE_NAME = "Surface.1"
SENSOR_NAME = "Irradiance.1"

## Model Setup

### Load assets
The assets used to run this example are available in the
[PySpeos repository](https://github.com/ansys/pyspeos/) on GitHub.

> **Note:** Make sure you
> have downloaded simulation assets and set ``assets_data_path``
> to point to the assets folder.

In [3]:
if USE_DOCKER:  # Running on the remote server.
    assets_data_path = Path("/app") / "assets"
else:
    assets_data_path = Path("/path/to/your/download/assets/directory")

### Connect to the RPC Server
This Python client connects to a server where the Speos engine
is running as a service. In this example, the server and
client are the same machine. The launch_local_speos_rpc_method can
be used to start a local instance of the service..

In [4]:
if USE_DOCKER:
    speos = Speos(channel=default_docker_channel())
else:
    speos = launcher.launch_local_speos_rpc_server(port=GRPC_PORT)

/home/runner/work/pyspeos/pyspeos/.venv/lib/python3.14/site-packages/ansys/tools/common/cyberchannel.py:188: UserWarning: Starting gRPC client without TLS on localhost:50098. This is INSECURE. Consider using a secure connection.
  warn(f"Starting gRPC client without TLS on {target}. This is INSECURE. Consider using a secure connection.")


### Create a new project

The only way to create a simulation using the core layer, is to create it from a
project. The ``Project`` class is instantiated by passing a ``Speos`` instance.

In [5]:
p = Project(speos=speos)
print(p)

{
    "name": "",
    "description": "",
    "metadata": {},
    "part_guid": "",
    "sub_scene_anchor_axis_system": [],
    "sources": [],
    "sensors": [],
    "simulations": [],
    "materials": [],
    "scenes": []
}


### Prepare prerequisites

Create the necessary elements for a simulation: Sensor, source, root part, optical property are
prerequisites.

### Prepare the root part

In [6]:
root_part = p.create_root_part()
body_1 = root_part.create_body(name="Body.1")
face_1 = body_1.create_face(name="Face.1")
face_1.vertices = [0, 1, 2, 0, 2, 2, 1, 2, 2]
face_1.facets = [0, 1, 2]
face_1.normals = [0, 0, 1, 0, 0, 1, 0, 0, 1]
root_part.commit()

### Prepare an optical property
Create Optical Property

In [7]:
opt_prop = p.create_optical_property("Material.1")
opt_prop.set_volume_opaque().set_surface_mirror()

Choose the geometry for this optical property : Body.1

In [8]:
opt_prop.geometries = [body_1]
opt_prop.commit()

### Prepare an irradiance sensor

In [9]:
sensor1 = p.create_sensor(name=SENSOR_NAME)
# set type to colorimetric or spectral so that the sensor can be used both in
# direct and inverse simulation
sensor1.set_type_colorimetric()
sensor1.commit()

### Prepare a surface source

In [10]:
source1 = p.create_source(name=SOURCE_NAME)
source1.set_exitance_constant().geometries = [(face_1, True)]
# define a spectrum which is not monochromatic so it can be used in both direct and inverse
# simulation
source1.spectrum.set_blackbody()
source1.commit()

/home/runner/work/pyspeos/pyspeos/.venv/lib/python3.14/site-packages/ansys/speos/core/project.py:393: UserWarning: The pySpeos feature : SourceSurface needs a Speos Version of 2025 R2 SP0 or higher.
  feature = SourceSurface(


## Create a simulation

In [11]:
simulation1 = p.create_simulation(name="Simulation.1")
simulation1.sensor_paths = [sensor1]  # use sensor instance object
simulation1.source_paths = [source1]  # use source instance object
print(simulation1)
simulation1.commit()
print(simulation1)

local: {
    "name": "Simulation.1",
    "sensor_paths": [
        "Irradiance.1"
    ],
    "source_paths": [
        "Surface.1"
    ],
    "geometries": {
        "geo_paths": []
    },
    "display_name": "",
    "description": "",
    "metadata": {},
    "simulation_guid": "",
    "simulation": {
        "direct_mc_simulation_template": {
            "geom_distance_tolerance": 0.01,
            "max_impact": 100,
            "weight": {
                "minimum_energy_percentage": 0.005
            },
            "dispersion": true,
            "colorimetric_standard": "CIE_1931",
            "fast_transmission_gathering": false,
            "ambient_material_uri": "",
            "stop_condition_rays_number": "200000",
            "automatic_save_frequency": 1800
        },
        "name": "Simulation.1",
        "description": "",
        "metadata": {},
        "scene_guid": "43ce5e92-ebf6-40c7-a521-f4d093565ef3",
        "simulation_path": "Simulation.1",
        "job_type": "

/home/runner/work/pyspeos/pyspeos/.venv/lib/python3.14/site-packages/ansys/speos/core/project.py:552: UserWarning: The pySpeos feature : SimulationDirect needs a Speos Version of 2025 R2 SP0 or higher.
  feature = SimulationDirect(
/home/runner/work/pyspeos/pyspeos/.venv/lib/python3.14/site-packages/ansys/speos/core/simulation.py:1074: UserWarning: Please note that setting a value for light expert option forces a sensorcommit when committing the Simulation class
  self.light_expert = default_parameters.light_expoert


### Set simulation characteristics

Simulation is defined with the same default values as the GUI speos.

If the user would like to modify the simulation characteristics,
it is possible to do so by setting the simulation characteristics as below.

In [12]:
simulation2_direct = p.create_simulation(name="Simulation.2")
simulation2_direct.ambient_material_file_uri = assets_data_path / "AIR.material"
simulation2_direct.set_colorimetric_standard_CIE_1964().set_weight_none().set_dispersion = False
simulation2_direct.geom_distance_tolerance = 0.01
simulation2_direct.max_impact = 200
simulation2_direct.sensor_paths = [SENSOR_NAME]  # use sensor instance name
simulation2_direct.source_paths = [SOURCE_NAME]  # use source instance name
simulation2_direct.commit()
print(simulation2_direct)

{
    "name": "Simulation.2",
    "metadata": {
        "UniqueId": "84a8c7e3-57d4-4b48-a452-7b98ec6b99e9"
    },
    "simulation_guid": "8c7fa327-d280-4844-9feb-36879e2914e4",
    "sensor_paths": [
        "Irradiance.1"
    ],
    "source_paths": [
        "Surface.1"
    ],
    "geometries": {
        "geo_paths": []
    },
    "display_name": "",
    "description": "",
    "simulation": {
        "direct_mc_simulation_template": {
            "geom_distance_tolerance": 0.01,
            "max_impact": 200,
            "colorimetric_standard": "CIE_1964",
            "dispersion": true,
            "ambient_material_uri": "/app/assets/AIR.material",
            "fast_transmission_gathering": false,
            "stop_condition_rays_number": "200000",
            "automatic_save_frequency": 1800
        },
        "name": "Simulation.2",
        "description": "",
        "metadata": {},
        "scene_guid": "43ce5e92-ebf6-40c7-a521-f4d093565ef3",
        "simulation_path": "Simulatio

### Read information

Read simulation information

In [13]:
print(simulation1)

{
    "name": "Simulation.1",
    "metadata": {
        "UniqueId": "69dfdb34-7a09-4d71-805b-fb671be028c0"
    },
    "simulation_guid": "94b0ffaf-6da8-4fca-9a19-0340219835df",
    "sensor_paths": [
        "Irradiance.1"
    ],
    "source_paths": [
        "Surface.1"
    ],
    "geometries": {
        "geo_paths": []
    },
    "display_name": "",
    "description": "",
    "simulation": {
        "direct_mc_simulation_template": {
            "geom_distance_tolerance": 0.01,
            "max_impact": 100,
            "weight": {
                "minimum_energy_percentage": 0.005
            },
            "dispersion": true,
            "colorimetric_standard": "CIE_1931",
            "fast_transmission_gathering": false,
            "ambient_material_uri": "",
            "stop_condition_rays_number": "200000",
            "automatic_save_frequency": 1800
        },
        "name": "Simulation.1",
        "description": "",
        "metadata": {},
        "scene_guid": "43ce5e92-e

Read project information

In [14]:
print(p)

{
    "part_guid": "1db09201-84ad-40f5-9888-c4743701fd59",
    "sources": [
        {
            "name": "Surface.1",
            "metadata": {
                "UniqueId": "d4210746-6bfd-46a4-a22c-ab1920a6d0f7"
            },
            "source_guid": "6a91a1d8-6e81-43ea-9c43-c2c38bc3a167",
            "display_name": "",
            "description": "",
            "source": {
                "name": "Surface.1",
                "surface": {
                    "luminous_flux": {
                        "luminous_value": 683.0
                    },
                    "intensity_guid": "82251d75-c3b1-4d4c-9556-26332ee4c3fd",
                    "exitance_constant": {
                        "geo_paths": [
                            {
                                "geo_path": "Body.1/Face.1",
                                "reverse_normal": true
                            }
                        ]
                    },
                    "spectrum_guid": "2ce0aea5-a2d2-4d28-8

## Update simulation settings

If you are manipulating a simulation already committed, remember to commit your changes.

If you don't, you will still only watch what is committed on the server.

In [15]:
simulation1.ambient_material_file_uri = assets_data_path / "AIR.material"
simulation1.commit()
print(simulation1)

{
    "name": "Simulation.1",
    "metadata": {
        "UniqueId": "69dfdb34-7a09-4d71-805b-fb671be028c0"
    },
    "simulation_guid": "94b0ffaf-6da8-4fca-9a19-0340219835df",
    "sensor_paths": [
        "Irradiance.1"
    ],
    "source_paths": [
        "Surface.1"
    ],
    "geometries": {
        "geo_paths": []
    },
    "display_name": "",
    "description": "",
    "simulation": {
        "direct_mc_simulation_template": {
            "geom_distance_tolerance": 0.01,
            "max_impact": 100,
            "weight": {
                "minimum_energy_percentage": 0.005
            },
            "dispersion": true,
            "ambient_material_uri": "/app/assets/AIR.material",
            "colorimetric_standard": "CIE_1931",
            "fast_transmission_gathering": false,
            "stop_condition_rays_number": "200000",
            "automatic_save_frequency": 1800
        },
        "name": "Simulation.1",
        "description": "",
        "metadata": {},
        "

## Reset

Possibility to reset local values from the one available in the server.

In [16]:
simulation1.max_impact = 1000  # adjust max impact but no commit
simulation1.reset()  # reset -> this will apply the server value to the local value
simulation1.delete()  # delete (to display the local value with the below print)
print(simulation1)

local: {
    "name": "Simulation.1",
    "sensor_paths": [
        "Irradiance.1"
    ],
    "source_paths": [
        "Surface.1"
    ],
    "geometries": {
        "geo_paths": []
    },
    "display_name": "",
    "description": "",
    "metadata": {},
    "simulation_guid": "",
    "simulation": {
        "direct_mc_simulation_template": {
            "geom_distance_tolerance": 0.01,
            "max_impact": 100,
            "weight": {
                "minimum_energy_percentage": 0.005
            },
            "dispersion": true,
            "ambient_material_uri": "/app/assets/AIR.material",
            "colorimetric_standard": "CIE_1931",
            "fast_transmission_gathering": false,
            "stop_condition_rays_number": "200000",
            "automatic_save_frequency": 1800
        },
        "name": "Simulation.1",
        "description": "",
        "metadata": {},
        "scene_guid": "43ce5e92-ebf6-40c7-a521-f4d093565ef3",
        "simulation_path": "Simulation.1

## Other simulation examples

### Inverse simulation

In [17]:
simulation3 = p.create_simulation(name="Simulation.3", feature_type=SimulationInverse)
simulation3.sensor_paths = [SENSOR_NAME]
simulation3.source_paths = [source1]
simulation3.commit()
print(simulation3)

/home/runner/work/pyspeos/pyspeos/.venv/lib/python3.14/site-packages/ansys/speos/core/project.py:567: UserWarning: The pySpeos feature : SimulationInverse needs a Speos Version of 2025 R2 SP0 or higher.
  feature = SimulationInverse(


{
    "name": "Simulation.3",
    "metadata": {
        "UniqueId": "cbb00029-170b-421a-8849-aae0ee5b8b38"
    },
    "simulation_guid": "7af0fd96-2786-41a1-aa0a-2fd5ae02a353",
    "sensor_paths": [
        "Irradiance.1"
    ],
    "source_paths": [
        "Surface.1"
    ],
    "geometries": {
        "geo_paths": []
    },
    "display_name": "",
    "description": "",
    "simulation": {
        "inverse_mc_simulation_template": {
            "geom_distance_tolerance": 0.01,
            "max_impact": 100,
            "weight": {
                "minimum_energy_percentage": 0.005
            },
            "number_of_gathering_rays_per_source": 1,
            "colorimetric_standard": "CIE_1931",
            "dispersion": false,
            "splitting": false,
            "maximum_gathering_error": 0,
            "maximum_gathering_error_percentage": 0.0,
            "fast_transmission_gathering": false,
            "ambient_material_uri": "",
            "optimized_propagation_none

### Interactive simulation

In [18]:
simulation4 = p.create_simulation(name="Simulation.4", feature_type=SimulationInteractive)
simulation4.source_paths = [source1]
simulation4.commit()
print(simulation4)

{
    "name": "Simulation.4",
    "metadata": {
        "UniqueId": "11cc0c7f-64ee-4af3-b4aa-0c6e78865770"
    },
    "simulation_guid": "7a80e7e5-21ee-4a73-b6e8-240b2bbb0d90",
    "source_paths": [
        "Surface.1"
    ],
    "geometries": {
        "geo_paths": []
    },
    "display_name": "",
    "description": "",
    "sensor_paths": [],
    "simulation": {
        "name": "Simulation.4",
        "interactive_simulation_template": {
            "geom_distance_tolerance": 0.01,
            "max_impact": 100,
            "weight": {
                "minimum_energy_percentage": 0.005
            },
            "colorimetric_standard": "CIE_1931",
            "ambient_material_uri": "",
            "rays_number_per_sources": [],
            "light_expert": false,
            "impact_report": false
        },
        "description": "",
        "metadata": {},
        "scene_guid": "43ce5e92-ebf6-40c7-a521-f4d093565ef3",
        "simulation_path": "Simulation.4",
        "job_type": 

### Virtual BSDF Bench simulation

In [19]:
# Change the material property from mirror to bsdf type
opt_prop.set_surface_library()
opt_prop.sop_library.file_uri = assets_data_path / "R_test.anisotropicbsdf"
opt_prop.commit()
vbb = p.create_simulation(name="virtual_BSDF", feature_type=SimulationVirtualBSDF)
vbb.axis_system = [
    0.36,
    1.73,
    2.0,
    1.0,
    0.0,
    0.0,
    0.0,
    1.0,
    0.0,
    0.0,
    0.0,
    1.0,
]  # change the coordinate VBSDF to body center
vbb.commit()
results = vbb.compute_CPU()
print(results)

[upload_response {
  info {
    uri: "ccb8782d-3a6d-426b-ba5f-3d0f96372114"
    file_name: "virtual_BSDF.html"
    file_size: 27131
  }
}
, upload_response {
  info {
    uri: "d415fe35-619f-4a0a-883d-a165a1cf7e34"
    file_name: "virtual_BSDF.anisotropicbsdf"
    file_size: 1260570
  }
}
]


## Simulation compute and stop

The compute_CPU, compute_GPU calls are blocking.
Thus it will return only once the simulation compute is finished.

If you want to have the possibility to stop it before it is finished, here an example.

In [20]:
from threading import Thread
from time import sleep

In [21]:
# Launch the compute_CPU in a thread and start the thread.
compute_thread = Thread(target=vbb.compute_CPU)
compute_thread.start()
# Wait 2 seconds then stop_computation
sleep(2)
vbb.stop_computation()
# Join the compute_thread
compute_thread.join()

In [22]:
speos.close()

True